In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
days = [
    "day_1",
    "day_2",
    "day_3",
    "day_4",
    "day_5"
]

silver_path = "dbfs:/EmployeeLifecycleProject/silver/"
history_path = "dbfs:/EmployeeLifecycleProject/history"

In [0]:
day1 = spark.read.format("delta").load(silver_path + days[0])

history = (
    day1
    .withColumn("effective_start_date", lit("day_1"))
    .withColumn("effective_end_date", lit(None).cast("string"))
    .withColumn("is_current", lit(True))
)

In [0]:
history.write \
    .format("delta") \
    .mode("overwrite") \
    .save(history_path)

In [0]:
history_df = spark.read.format("delta").load(history_path)
display(history_df)

emp_id,name,department,salary,join_date,status,effective_start_date,effective_end_date,is_current
151,Emp_151,MARKETING,73963,2020-03-08,ACTIVE,day_1,null,true
152,Emp_152,MARKETING,79253,2020-03-31,ACTIVE,day_1,null,true
153,Emp_153,IT,83668,2021-10-16,ACTIVE,day_1,null,true
154,Emp_154,FINANCE,84331,2020-06-28,ACTIVE,day_1,null,true
155,Emp_155,MARKETING,90570,2021-12-28,ACTIVE,day_1,null,true
156,Emp_156,MARKETING,47947,2020-07-23,ACTIVE,day_1,null,true
157,Emp_157,HR,80039,2020-10-23,ACTIVE,day_1,null,true
158,Emp_158,MARKETING,94302,2020-02-13,ACTIVE,day_1,null,true
159,Emp_159,FINANCE,69834,2021-05-20,ACTIVE,day_1,null,true
160,Emp_160,MARKETING,90745,2020-08-17,ACTIVE,day_1,null,true


In [0]:
for i in range(1, len(days)):

    current_day = days[i]

    print("=" * 60)
    print(f"Processing {current_day}")
    print("=" * 60)

    # Read current day's Silver snapshot
    current_df = spark.read.format("delta").load(silver_path + current_day)

    # Read latest history
    history_df = spark.read.format("delta").load(history_path)

    # Only current records
    current_history = history_df.filter(col("is_current") == True)

    # Join history with today's snapshot
    joined_df = current_history.alias("h").join(
        current_df.alias("c"),
        "emp_id",
        "inner"
    )

    # Detect changed employees
    changed_df = joined_df.filter(
        (col("h.department") != col("c.department")) |
        (col("h.salary") != col("c.salary")) |
        (col("h.status") != col("c.status"))
    )

    changed_ids = [r.emp_id for r in changed_df.select("emp_id").collect()]

    # ------------------------------------------------------------------
    # Expire current records
    # ------------------------------------------------------------------
    if changed_ids:

        expired = (
            history_df
            .withColumn(
                "effective_end_date",
                when(
                    (col("emp_id").isin(changed_ids)) &
                    (col("is_current") == True),
                    current_day
                ).otherwise(col("effective_end_date"))
            )
            .withColumn(
                "is_current",
                when(
                    (col("emp_id").isin(changed_ids)) &
                    (col("is_current") == True),
                    False
                ).otherwise(col("is_current"))
            )
        )

        expired.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(history_path)

    # Reload history after expiring rows
    history_df = spark.read.format("delta").load(history_path)

    # ------------------------------------------------------------------
    # Insert updated records
    # ------------------------------------------------------------------
    updated_records = (
        changed_df
        .select(
            col("c.emp_id").alias("emp_id"),
            col("c.name").alias("name"),
            col("c.department").alias("department"),
            col("c.salary").alias("salary"),
            col("c.join_date").alias("join_date"),
            col("c.status").alias("status")
        )
        .withColumn("effective_start_date", lit(current_day))
        .withColumn("effective_end_date", lit(None).cast("string"))
        .withColumn("is_current", lit(True))
    )

    if updated_records.count() > 0:

        updated_records.write \
            .format("delta") \
            .mode("append") \
            .save(history_path)

    # ------------------------------------------------------------------
    # Refresh current history
    # ------------------------------------------------------------------
    history_df = spark.read.format("delta").load(history_path)

    current_history = history_df.filter(col("is_current") == True)

    # ------------------------------------------------------------------
    # Find new employees
    # ------------------------------------------------------------------
    new_employees = (
        current_df.alias("c")
        .join(
            current_history.alias("h"),
            "emp_id",
            "left_anti"
        )
    )

    new_employees = (
        new_employees
        .withColumn("effective_start_date", lit(current_day))
        .withColumn("effective_end_date", lit(None).cast("string"))
        .withColumn("is_current", lit(True))
    )

    if new_employees.count() > 0:

        new_employees.write \
            .format("delta") \
            .mode("append") \
            .save(history_path)

    print(f"{current_day} completed")

Processing day_2
day_2 completed
Processing day_3
day_3 completed
Processing day_4
day_4 completed
Processing day_5
day_5 completed


In [0]:
history_final = spark.read.format("delta").load(history_path)
display(
    history_final.orderBy(
        "emp_id",
        "effective_start_date"
    )
)

emp_id,name,department,salary,join_date,status,effective_start_date,effective_end_date,is_current
1,Emp_1,SALES,46632,2020-07-30,ACTIVE,day_1,day_2,false
1,Emp_1,MARKETING,46632,2020-07-30,ACTIVE,day_2,day_3,false
1,Emp_1,FINANCE,46632,2020-07-30,ACTIVE,day_3,day_5,false
1,Emp_1,FINANCE,54780,2020-07-30,ACTIVE,day_5,null,true
2,Emp_2,HR,65513,2021-06-19,ACTIVE,day_1,day_2,false
2,Emp_2,FINANCE,65513,2021-06-19,ACTIVE,day_2,day_4,false
2,Emp_2,SALES,65513,2021-06-19,ACTIVE,day_4,null,true
3,Emp_3,SALES,84005,2020-09-01,ACTIVE,day_1,day_2,false
3,Emp_3,SALES,93010,2020-09-01,ACTIVE,day_2,day_4,false
3,Emp_3,IT,93010,2020-09-01,ACTIVE,day_4,null,true


In [0]:
history_final.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/history")